# Week 6 — 財務數學與選擇權定價

> 本 notebook 屬於「量化數學路線圖」（quant-math-roadmap）開源教學專案。
> 僅供**教育與研究方法論**用途，**不構成投資建議**，任何結果都不代表實際可獲利或可投資的策略。

## 學習目標

- 計算折現因子、現值與債券價格。
- 繪製 call、put 與簡單組合的 payoff 圖。
- 用 binomial tree 為歐式選擇權定價。
- 對 strike、波動度、到期、利率做敏感度分析。

## 預估學習時間

約 8–10 小時。

## 先備概念

- 基本代數與指數
- Week 1 的報酬概念

## 外部學習資源

- [NTU OpenCourseWare 基礎財金素養](https://ocw.aca.ntu.edu.tw/courses/110S204)

> 外部資源僅供參考連結；本專案不重製任何受版權保護的課程材料。

## 概念說明

### 貨幣的時間價值

未來的現金流要先**折現**才能和今天的錢比較。年利率 $r$、$t$ 年後、每年複利 $m$ 次的折現因子為 $(1 + r/m)^{-mt}$。現值是各期現金流乘上折現因子後的總和。

### 選擇權 payoff 與 binomial 定價

歐式 call/put 到期 payoff 為 $\max(S-K,0)$、$\max(K-S,0)$。

本路線圖**刻意只用 binomial tree**：它只需要算術與 no-arbitrage 概念，**不需要** stochastic calculus，也**不要求** Black-Scholes 推導。選擇權價值是其到期 payoff 在 risk-neutral 機率下的折現期望值。

> 提醒：模型價格**不應**被解讀為對真實市場價格的預測。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quant_math_roadmap.finance.fixed_income import (
    bond_price, discount_factor, present_value, zero_coupon_bond_price,
)
from quant_math_roadmap.finance.derivatives import (
    binomial_european_option, call_payoff, put_payoff,
    long_straddle_payoff, put_call_parity_gap,
)

### 現值計算器

In [ ]:
rate = 0.04
cash_flows = [100, 100, 100, 1100]  # 4 年期、年付息
times = [1, 2, 3, 4]
pv = present_value(cash_flows, times, rate)
print(f'折現率 {rate:.0%} 下，現金流現值 = {pv:.2f}')
for t in times:
    print(f'  t={t}: 折現因子 = {discount_factor(rate, t):.4f}')

### 債券定價：價格隨殖利率下降

In [ ]:
yields = np.linspace(0.01, 0.10, 50)
prices = [bond_price(1000, 0.05, 10, y, coupons_per_year=2) for y in yields]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(yields, prices, label='10 年期、5% 票息債券')
ax.axhline(1000, linestyle='--', label='面額 = 1000')
ax.set_title('債券價格 vs 殖利率')
ax.set_xlabel('殖利率 (yield to maturity)')
ax.set_ylabel('債券價格')
ax.legend()
plt.show()
zcb = zero_coupon_bond_price(1000, 10, 0.05)
print(f'10 年期零息債券（殖利率 5%）價格 = {zcb:.2f}')

殖利率上升，債券價格下降；票息率等於殖利率時，債券以面額（par）定價。

### Payoff 圖

In [ ]:
spot_grid = np.linspace(50, 150, 200)
strike = 100.0
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].plot(spot_grid, call_payoff(spot_grid, strike))
axes[0].set_title('Call payoff (K=100)')
axes[1].plot(spot_grid, put_payoff(spot_grid, strike))
axes[1].set_title('Put payoff (K=100)')
axes[2].plot(spot_grid, long_straddle_payoff(spot_grid, strike))
axes[2].set_title('Long straddle payoff (K=100)')
for ax in axes:
    ax.set_xlabel('到期標的價格 S')
    ax.set_ylabel('payoff')
plt.tight_layout()
plt.show()

### Binomial 歐式選擇權定價

In [ ]:
params = {'spot': 100.0, 'strike': 100.0, 'rate': 0.05,
          'volatility': 0.20, 'maturity': 1.0}
call = binomial_european_option(**params, n_steps=300, option_type='call')
put = binomial_european_option(**params, n_steps=300, option_type='put')
print(f'歐式 call 價格 = {call:.4f}')
print(f'歐式 put  價格 = {put:.4f}')
gap = put_call_parity_gap(call, put, params['spot'], params['strike'],
                          params['rate'], params['maturity'])
print(f'put-call parity 殘差 = {gap:.6f}  (應接近 0)')

### 敏感度分析

In [ ]:
vols = np.linspace(0.05, 0.6, 40)
call_by_vol = [binomial_european_option(100, 100, 0.05, v, 1.0,
               n_steps=200) for v in vols]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(vols, call_by_vol, label='ATM call (S=K=100)')
ax.set_title('歐式 call 價格 vs 波動度')
ax.set_xlabel('波動度代理值 sigma')
ax.set_ylabel('call 價格')
ax.legend()
plt.show()
print('波動度越高，選擇權越貴 — 因為更大的不確定性對買方有利。')

## 練習

請依序完成以下練習。**基礎練習**鞏固定義，**應用練習**動手寫程式，**反思問題**把數學連結到回測與研究方法論。

> 主 notebook 的程式練習提供可執行的起始碼（starter）。完整參考解答請見 `notebooks/solutions/` 對應的 `_solution` notebook。

### 基礎練習

1. 用一句話解釋為什麼未來的錢要折現。
2. 為什麼債券價格與殖利率反向變動？
3. 解釋 long straddle 的 payoff 形狀，以及它在押注什麼。

### 應用練習

In [ ]:
# 應用練習 1：計算 binomial call 價格隨 strike 變化的曲線（其他參數固定），
# 並確認 strike 越高、call 越便宜。
strikes = np.linspace(80, 120, 20)
call_by_strike = None  # TODO: [binomial_european_option(100, k, 0.05, 0.2, 1.0, n_steps=150) for k in strikes]
if call_by_strike is not None:
    print('遞減?', all(np.diff(call_by_strike) < 0))

In [ ]:
# 應用練習 2：驗證 binomial 步數增加時 call 價格收斂（穩定下來）。
for steps in [10, 50, 200, 800]:
    price = None  # TODO: binomial_european_option(100, 100, 0.05, 0.2, 1.0, n_steps=steps)
    print(steps, price)

### 反思問題

1. binomial 模型價格與真實市場價格幾乎不會完全相同。這對「用模型價格設計交易策略」有什麼提醒？

## 常見錯誤

- **折現時搞錯複利頻率（年複利 vs 半年複利）。**
- **把選擇權 payoff（到期才實現）與選擇權現價混為一談。**
- **binomial 步數太少就把價格當精確值。**
- **宣稱模型價格等於真實市場價格。**

## 完成本週後，你應該能做到什麼

- [ ] 能計算現值與債券價格。
- [ ] 能繪製並解讀 call/put/straddle 的 payoff 圖。
- [ ] 能用 binomial tree 為歐式選擇權定價並解釋每一步。
- [ ] 能對主要參數做敏感度分析。

## 參考與致謝

- 本 notebook 的所有解說、範例與習題皆為本專案**原創**撰寫。
- 推薦的外部學習資源請見 [`docs/resources.md`](../docs/resources.md)。
- 數學與財務概念筆記請見 [`docs/math/`](../docs/math/) 與 [`docs/finance/`](../docs/finance/)。

### 隱私與免責聲明

- 本 notebook 不含任何真實個人資訊。
- 本 notebook 僅使用可重現的合成資料，不需要網路連線。
- 本 notebook 不對任何策略做出實際投資獲利的宣稱。